# Titanic — XGBoost

Focused notebook: XGBoost with fixed params, engineered features.

| Param | Value |
|---|---|
| `colsample_bytree` | 0.8 |
| `learning_rate` | 0.01 |
| `max_depth` | 3 |
| `n_estimators` | 300 |
| `subsample` | 1.0 |

## Kaggle Results

| # | Score |
|---|---|
| 1 | — |

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

print(train_data.shape, test_data.shape)

In [ ]:
# Title extraction and group-based Age imputation
train_data['Title'] = train_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
train_data['Title'] = train_data['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
)
train_data['Title'] = train_data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

title_age_map = train_data.groupby('Title')['Age'].median()
train_data['Age'] = train_data['Age'].fillna(train_data['Title'].map(title_age_map))
train_data['Title'] = train_data['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})

train_data['Embarked'] = train_data['Embarked'].fillna(train_data['Embarked'].mode()[0])
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1})
train_data['Embarked'] = train_data['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
train_data['IsAlone'] = (train_data['FamilySize'] == 1).astype(int)

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
            'Title', 'FamilySize', 'IsAlone']

X = train_data[features]
y = train_data['Survived']

model = XGBClassifier(
    colsample_bytree=0.8,
    learning_rate=0.01,
    max_depth=3,
    n_estimators=300,
    subsample=1.0,
    random_state=42,
    eval_metric='logloss'
)

cv_score = cross_val_score(model, X, y, cv=5, scoring='accuracy', n_jobs=-1).mean()
print(f'5-fold CV accuracy: {cv_score:.4f}')

model.fit(X, y)

In [ ]:
# Apply same preprocessing to test data using train-derived statistics
test_data['Title'] = test_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test_data['Title'] = test_data['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
)
test_data['Title'] = test_data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

test_data['Age'] = test_data['Age'].fillna(test_data['Title'].map(title_age_map))
test_data['Age'] = test_data['Age'].fillna(title_age_map.median())
test_data['Title'] = test_data['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})

test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].median())
test_data['Embarked'] = test_data['Embarked'].fillna(test_data['Embarked'].mode()[0])
test_data['Sex'] = test_data['Sex'].map({'male': 0, 'female': 1})
test_data['Embarked'] = test_data['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1
test_data['IsAlone'] = (test_data['FamilySize'] == 1).astype(int)

X_test = test_data[features].fillna(test_data[features].median())

predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Survived': predictions})
output.to_csv('submission_xgb.csv', index=False)
print('Saved submission_xgb.csv')